# BEAM Linkstats Consist Analysis

This notebook bootstraps a Consist tracker against an **archived PILATES run** (scratch storage), lists linkstats artifacts by facets, and computes first-pass summary + delta metrics over phys-sim iterations.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from pilates.utils.consist_analysis import (
    create_analysis_tracker,
    find_linkstats_artifacts,
    summarize_linkstats_artifacts,
    summarize_linkstats_deltas,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# --- Configure these for your environment/run ---
USER = os.environ.get("USER", "$USER")
RUN_NAME = "new-asim-consist-flash-20260209-103617"

OUTPUT_ROOT = Path(f"/global/scratch/users/{USER}/pilates-output")
PROJECT_ROOT = Path(f"/global/scratch/users/{USER}/sources/PILATES")
ARCHIVE_RUN_DIR = OUTPUT_ROOT / RUN_NAME

# If your DB lives elsewhere, set it explicitly (for example scratch/consist-seattle.duckdb).
DB_PATH = ARCHIVE_RUN_DIR / "provenance.duckdb"

TARGET_YEAR = 2018
TARGET_ITERATION = 0

print("DB_PATH:", DB_PATH)
print("ARCHIVE_RUN_DIR:", ARCHIVE_RUN_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
tracker = create_analysis_tracker(
    db_path=DB_PATH,
    archive_run_dir=ARCHIVE_RUN_DIR,
    project_root=PROJECT_ROOT,
    output_root=OUTPUT_ROOT,
)

print("Tracker mounts:")
for name, root in tracker.mounts.items():
    print(f"  {name}:// -> {root}")

In [ ]:
# Primary query: phys-sim unmodified linkstats parquet artifacts for one year/iteration
artifacts_df = find_linkstats_artifacts(
    tracker,
    year=TARGET_YEAR,
    iteration=TARGET_ITERATION,
    artifact_family="linkstats_unmodified_phys_sim_iter_parquet",
    namespace="beam",
    limit=20000,
)

print(f"Found {len(artifacts_df)} artifacts")
artifacts_df.head(20)

In [ ]:
# Quick facet/view sanity check
if artifacts_df.empty:
    print("No linkstats artifacts found with the requested filters.")
else:
    display(
        artifacts_df[[
            "key",
            "container_uri",
            "facet_source",
            "phys_sim_iteration",
            "beam_sub_iteration",
        ]].head(25)
    )
    print("Facet source summary:")
    display(artifacts_df["facet_source"].value_counts(dropna=False))


In [ ]:
summary_df = summarize_linkstats_artifacts(artifacts_df, tracker=tracker)
print(f"Computed summaries for {len(summary_df)} artifacts via Consist views")
summary_df.head(20)

In [ ]:
# First-pass rollup by sub-iteration + phys-sim iteration
if summary_df.empty:
    print("No summary rows available. Check mounts/path resolution above.")
else:
    rollup = (
        summary_df.groupby(["year", "iteration", "beam_sub_iteration", "phys_sim_iteration"], dropna=False)
        .agg(
            artifacts=("key", "count"),
            rows_total=("row_count", "sum"),
            distinct_links_mean=("distinct_links", "mean"),
            volume_sum_total=("volume_sum", "sum"),
            traveltime_mean_mean=("traveltime_mean", "mean"),
            traveltime_p95_mean=("traveltime_p95", "mean"),
        )
        .reset_index()
        .sort_values(["year", "iteration", "beam_sub_iteration", "phys_sim_iteration"], na_position="last")
    )
    display(rollup)

In [ ]:
# Consecutive phys-sim deltas within each (year, iteration, beam_sub_iteration) sequence
delta_df = summarize_linkstats_deltas(summary_df, tracker=tracker)
print(f"Computed {len(delta_df)} delta rows")
delta_df

## Notes

- This notebook intentionally mounts `workspace://` to the archived run directory.
- It computes metrics from Consist hybrid views (`tracker.create_view(...)`) rather than direct parquet paths.
- If view creation fails, verify mounts and ensure your Consist DB schema matches your Consist package version.
- You can switch artifact family to `linkstats_parquet` or `linkstats` in `find_linkstats_artifacts(...)` for other analyses.
